# Práctica 2 — Secciones 3.4 y 3.5
## Detección de bordes y detección de esquinas

**Responsable:** Brigite

Este notebook continúa el pipeline a partir de la salida de operaciones morfológicas
(sección 3.3, `practica_2_playground.ipynb`).

**Entrada:** `muestra_procesada`, lista de tuplas `(ruta, clase, bgr, mascara)`.

**Salida:** figuras en `figs/` para el informe y la presentación.

---
### Contenido
1. Configuración y reconstrucción del pipeline base
2. Ajuste de máscara (erosión compensatoria)
3. **3.4** — Sobel y Canny
4. **3.5** — Harris y Shi-Tomasi
5. Análisis cuantitativo: contorno vs. interior de la hoja
6. Conclusiones

## 1. Configuración y pipeline base

> Las funciones de esta sección son una **copia fiel** de la celda 3 de
> `practica_2_playground.ipynb`. Si el pipeline cambia, actualizar `CONFIG_PIPELINE`.
>
> (Mejora pendiente para el grupo: mover estas funciones a un `pipeline.py`
> que ambos notebooks importen, para no duplicar código.)

In [ ]:
import os, random
import numpy as np
import cv2
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

# ---- Dataset ----
LOCAL_DATASET = False   # True si ya lo tienes descargado localmente

if LOCAL_DATASET:
    home = os.path.expanduser("~")
    path = os.path.join(home, "Downloads", "archive")  # ajustar a tu ruta
else:
    import kagglehub
    path = kagglehub.dataset_download("emmarex/plantdisease")

DATA_DIR = os.path.join(path, "PlantVillage")
print("DATA_DIR:", DATA_DIR)

# ---- Carpeta de salida para figuras ----
FIGS = "figs"
os.makedirs(FIGS, exist_ok=True)

def guardar(nombre):
    """Guarda la figura actual en figs/ con buena resolución."""
    ruta = os.path.join(FIGS, nombre)
    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    print("guardado:", ruta)

In [ ]:
# ============================================================
# COPIA del pipeline base (practica_2_playground.ipynb, celda 3)
# ============================================================

def to_grayscale(bgr):
    """Escala de grises por luminosidad (ITU-R BT.601)."""
    b, g, r = bgr[:, :, 0], bgr[:, :, 1], bgr[:, :, 2]
    return (0.299 * r + 0.587 * g + 0.114 * b).astype(np.uint8)

def aplicar_filtro(gray, filtro, ksize):
    if filtro == 'gaussian':
        return cv2.GaussianBlur(gray, (ksize, ksize), 0)
    elif filtro == 'median':
        return cv2.medianBlur(gray, ksize)
    raise ValueError("filtro debe ser 'gaussian' o 'median'")

def cdf_histograma(gray):
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).ravel()
    cdf = np.cumsum(hist)
    return (cdf - cdf.min()) / (cdf.max() - cdf.min()) * 255

def ecualizar_histograma(gray):
    lut = cdf_histograma(gray).astype(np.uint8)
    return cv2.LUT(gray, lut)

def orient_leaf_as_foreground(mask):
    """Deja la hoja en blanco (asume que el fondo domina el borde de la imagen)."""
    border = np.concatenate([mask[0, :], mask[-1, :], mask[:, 0], mask[:, -1]])
    return cv2.bitwise_not(mask) if border.mean() > 100 else mask

def segmentar(gray, metodo, global_t=127):
    if metodo == 'global':
        _, m = cv2.threshold(gray, global_t, 255, cv2.THRESH_BINARY)
        return orient_leaf_as_foreground(m)
    elif metodo == 'otsu':
        _, m = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
        return orient_leaf_as_foreground(m)
    raise ValueError("método desconocido: " + str(metodo))

def aplicar_morfologia(mask, operacion, ksize):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    if operacion == 'erosion':
        return cv2.erode(mask, kernel, iterations=1)
    elif operacion == 'dilatacion':
        return cv2.dilate(mask, kernel, iterations=1)
    elif operacion == 'apertura':
        return cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    elif operacion == 'cierre':
        return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    raise ValueError("operación desconocida: " + str(operacion))

def construir_pipeline(filtro='gaussian', ksize_filtro=5, contraste=False,
                       metodo_seg='otsu', global_t=127, operaciones_morf=()):
    pasos = [('Gris', to_grayscale)]
    if contraste:
        pasos.append(('Contraste (CDF)', ecualizar_histograma))
    pasos.append(('Filtro', lambda img: aplicar_filtro(img, filtro, ksize_filtro)))
    pasos.append(('Segmentacion', lambda img: segmentar(img, metodo_seg, global_t)))
    for op, ksize in operaciones_morf:
        pasos.append((op.capitalize() + " (k=" + str(ksize) + ")",
                      lambda img, op=op, ksize=ksize: aplicar_morfologia(img, op, ksize)))
    return pasos

def ejecutar_pipeline(bgr, pasos):
    actual = bgr
    for nombre, fn in pasos:
        actual = fn(actual)
    return actual

print("Funciones del pipeline listas.")

In [ ]:
# --- Configuración del pipeline final (P2, acordado por el grupo) ---
CONFIG_PIPELINE = dict(
    filtro='gaussian',
    ksize_filtro=5,
    contraste=True,
    metodo_seg='otsu',
    operaciones_morf=[('dilatacion', 11), ('erosion', 3)],
)

PIPELINE_FINAL = construir_pipeline(**CONFIG_PIPELINE)

def procesar_hasta_operaciones_morfologicas(ruta):
    """Corre el pipeline final sobre una imagen.
    Devuelve (bgr_original, mascara_binaria)."""
    bgr = cv2.imread(ruta)
    mascara = ejecutar_pipeline(bgr, PIPELINE_FINAL)
    return bgr, mascara

print("Pipeline:", " -> ".join(n for n, _ in PIPELINE_FINAL))

### Selección de imágenes de muestra

No necesitamos las 15 clases. Para las figuras del informe bastan **una hoja sana y
tres enfermas** con patrones visualmente distintos:

| Clase | Por qué |
|---|---|
| `Tomato_healthy` | Control: sin lesiones, superficie uniforme |
| `Tomato_Early_blight` | Lesiones grandes con anillos concéntricos |
| `Tomato_Septoria_leaf_spot` | Muchas manchas pequeñas y bien delimitadas |
| `Tomato__Tomato_YellowLeaf__Curl_Virus` | Deformación del contorno, sin manchas focales |

> **Coordinación:** usar estas mismas clases en todas las secciones del informe
> (segmentación, bordes/esquinas, LBP/HOG) para que se lea como un pipeline único.

In [ ]:
def get_class_dirs(data_dir=DATA_DIR):
    return sorted(d for d in os.listdir(data_dir)
                  if os.path.isdir(os.path.join(data_dir, d))
                  and d.lower() != "plantvillage")

DISPONIBLES = get_class_dirs()
print("Clases en el dataset:", len(DISPONIBLES))
for d in DISPONIBLES:
    print("  -", d)

In [ ]:
# Los nombres de carpeta tienen guiones bajos inconsistentes,
# así que resolvemos por coincidencia parcial en vez de hardcodear.

PATRONES = [
    "Tomato_healthy",
    "Tomato_Early_blight",
    "Tomato_Septoria",
    "YellowLeaf",
]

def resolver_clases(patrones, disponibles=DISPONIBLES):
    elegidas = []
    for p in patrones:
        match = [c for c in disponibles if p.lower() in c.lower()]
        if match:
            elegidas.append(match[0])
        else:
            print("[!] no se encontró la clase:", p)
    return elegidas

CLASES_FOCO = resolver_clases(PATRONES)
print("\nClases seleccionadas:")
for c in CLASES_FOCO:
    print("  -", c)

def es_sana(clase):
    return "healthy" in clase.lower()

def etiqueta_corta(clase):
    """Nombre legible para los títulos de las figuras."""
    return (clase.replace("Tomato_", "").replace("Tomato__", "")
                 .replace("_", " ").replace("  ", " ").strip())

In [ ]:
def imagenes_de_clase(clase, data_dir=DATA_DIR, n=1, seed=42):
    random.seed(seed)
    cdir = os.path.join(data_dir, clase)
    exts = ('.png', '.jpg', '.jpeg', '.bmp')
    imgs = sorted(f for f in os.listdir(cdir) if f.lower().endswith(exts))
    return [os.path.join(cdir, f) for f in random.sample(imgs, min(n, len(imgs)))]

# ---- Construcción de muestra_procesada ----
# Mismo formato que en el notebook de la sección 3.3: (ruta, clase, bgr, mascara)
N_POR_CLASE = 1

muestra_procesada = []
for clase in CLASES_FOCO:
    for ruta in imagenes_de_clase(clase, n=N_POR_CLASE):
        bgr, mascara = procesar_hasta_operaciones_morfologicas(ruta)
        muestra_procesada.append((ruta, clase, bgr, mascara))

print("Imágenes procesadas:", len(muestra_procesada))
for ruta, clase, bgr, mascara in muestra_procesada:
    print("  ", etiqueta_corta(clase), "->", bgr.shape, "| máscara:",
          round(100 * (mascara > 0).mean(), 1), "% de la imagen")

## 2. Estrategia de enmascarado y canal de análisis

Antes de detectar bordes hay que resolver dos decisiones de diseño. Ambas afectan
mucho el resultado y conviene justificarlas en el informe.

### 2.1 Cómo aplicar la máscara

Enmascarar la imagen en color y luego correr Canny **introduce un artefacto**: el
salto entre la hoja y el negro del fondo enmascarado genera un borde falso, y como la
máscara del pipeline está dilatada (`dilatación k=11` seguida de `erosión k=3` hace
crecer la región unos 4 px), ese borde falso queda *fuera* del contorno real y aparece
un contorno doble.

La estrategia correcta es calcular los bordes sobre la **imagen completa** y aplicar la
máscara **al resultado**. Así el contorno detectado es el de la hoja real y solo se
descarta el fondo lejano.

### 2.2 Sobre qué canal

La conversión a escala de grises es problemática aquí. El tejido verde sano y las
lesiones marrones tienen **luminancia parecida**, de modo que la diferencia se pierde
al pasar a gris:

| Región | gris | a\* de Lab | exceso de verde |
|---|---|---|---|
| Hoja sana | 84 | 94 | 120 |
| Lesión *early blight* | 98 (Δ**14**) | 133 (Δ**39**) | 10 (Δ**110**) |
| Lesión *septoria* | 153 (Δ69) | 131 (Δ37) | 15 (Δ105) |
| Clorosis (amarilleo) | 178 (Δ94) | 117 (Δ23) | 120 (Δ**0**) |

Con Δ = 14 niveles, Canny con umbral inferior en ~57 no detecta las lesiones de early
blight. El canal **a\*** de Lab codifica el eje verde–rojo y triplica ese contraste;
el exceso de verde separa aún mejor las lesiones necróticas pero es ciego al
amarilleo. Se toma **a\*** como opción por defecto por ser el más equilibrado entre
los tres tipos de síntoma.

In [ ]:
K_COMPENSA = 9      # compensa el crecimiento neto de dilatación(11) + erosión(3)
CANAL = 'lab_a'     # 'gris' | 'lab_a' | 'exceso_verde'

def ajustar_mascara(mascara, k=K_COMPENSA):
    """Erosión compensatoria: máscara ceñida al contorno real de la hoja."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    return cv2.erode(mascara, kernel, iterations=1)

def enmascarar(bgr, mascara):
    return cv2.bitwise_and(bgr, bgr, mask=mascara)

def canal_analisis(bgr, modo=None):
    """Canal de una banda sobre el que se calculan bordes y esquinas."""
    modo = modo or CANAL
    if modo == 'gris':
        return to_grayscale(bgr)
    if modo == 'lab_a':
        # a* de CIELAB: eje verde(-) <-> rojo(+); OpenCV lo entrega desplazado a [0,255]
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)[:, :, 1]
    if modo == 'exceso_verde':
        b = bgr[:, :, 0].astype(np.int16)
        g = bgr[:, :, 1].astype(np.int16)
        r = bgr[:, :, 2].astype(np.int16)
        return np.clip(2 * g - r - b, 0, 255).astype(np.uint8)
    raise ValueError("canal desconocido: " + str(modo))

print("Canal por defecto:", CANAL, "| erosión compensatoria k =", K_COMPENSA)

In [ ]:
# --- Figura 2.1: por qué no se enmascara antes de Canny ---
ruta, clase, bgr, mascara = muestra_procesada[1]
mascara_aj = ajustar_mascara(mascara)

ch_full = canal_analisis(bgr)
ch_negro = canal_analisis(enmascarar(bgr, mascara))          # enmascarado ANTES

e_sin    = cv2.Canny(ch_full, 50, 150)                        # sin máscara
e_antes  = cv2.Canny(ch_negro, 50, 150)                       # máscara antes (artefacto)
e_desp   = cv2.bitwise_and(cv2.Canny(ch_full, 50, 150),
                           cv2.Canny(ch_full, 50, 150), mask=mascara)  # máscara después

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for a, img, t in zip(axes,
        [cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), e_sin, e_antes, e_desp],
        ["Original",
         "(a) Sin mascara\n(ruido de fondo)",
         "(b) Mascara ANTES de Canny\n(contorno doble)",
         "(c) Mascara DESPUES de Canny\n(estrategia adoptada)"]):
    a.imshow(img, cmap=None if img.ndim == 3 else 'gray')
    a.set_title(t, fontsize=11); a.axis('off')
plt.suptitle("Estrategia de enmascarado — " + etiqueta_corta(clase), fontsize=14)
plt.tight_layout()
guardar("00_estrategia_mascara.png")
plt.show()

In [ ]:
# --- Figura 2.2: comparación de canales ---
CANALES = ['gris', 'lab_a', 'exceso_verde']
NOMBRES = {'gris': 'Escala de grises', 'lab_a': 'a* de Lab',
           'exceso_verde': 'Exceso de verde (2G-R-B)'}

enfermas = [m for m in muestra_procesada if not es_sana(m[1])][:2]

fig, axes = plt.subplots(len(enfermas), 7, figsize=(21, 3.6 * len(enfermas)))
if len(enfermas) == 1:
    axes = axes[np.newaxis, :]

for i, (ruta, clase, bgr, mascara) in enumerate(enfermas):
    axes[i, 0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title("Original" if i == 0 else "", fontsize=10)
    axes[i, 0].axis('off')
    axes[i, 0].text(-0.08, 0.5, etiqueta_corta(clase), transform=axes[i, 0].transAxes,
                    rotation=90, va='center', ha='center', fontsize=9)
    for j, cn in enumerate(CANALES):
        ch = canal_analisis(bgr, cn)
        e = cv2.bitwise_and(cv2.Canny(ch, 50, 150), cv2.Canny(ch, 50, 150), mask=mascara)
        axes[i, 1 + 2*j].imshow(ch, cmap='gray')
        axes[i, 1 + 2*j].set_title(NOMBRES[cn] if i == 0 else "", fontsize=10)
        axes[i, 2 + 2*j].imshow(e, cmap='gray')
        axes[i, 2 + 2*j].set_title("Canny sobre " + cn if i == 0 else "", fontsize=10)
        axes[i, 1 + 2*j].axis('off'); axes[i, 2 + 2*j].axis('off')

plt.suptitle("Elección del canal de análisis", fontsize=15, y=1.003)
plt.tight_layout()
guardar("01_comparacion_canales.png")
plt.show()

## 3. Sección 3.4 — Detección de bordes

**Operador de Sobel.** Aproxima las derivadas parciales mediante convolución con dos
kernels 3x3, uno horizontal y uno vertical. La magnitud del gradiente es
`sqrt(gx^2 + gy^2)`. Da respuesta continua y bordes de varios píxeles de ancho.

**Detector de Canny.** Cuatro etapas: suavizado gaussiano, cálculo del gradiente
(internamente Sobel), supresión de no-máximos —que adelgaza los bordes a un píxel— e
histéresis con dos umbrales, que conserva bordes débiles solo si están conectados a
bordes fuertes.

Los umbrales de Canny se derivan de la mediana calculada **únicamente sobre los píxeles
de la hoja**, para que no la arrastre el fondo gris del dataset.

In [ ]:
def sobel_magnitud(ch, ksize=3):
    """Magnitud del gradiente por Sobel, normalizada a uint8."""
    gx = cv2.Sobel(ch, cv2.CV_64F, 1, 0, ksize=ksize)
    gy = cv2.Sobel(ch, cv2.CV_64F, 0, 1, ksize=ksize)
    mag = np.sqrt(gx**2 + gy**2)
    if mag.max() > 0:
        mag = mag / mag.max() * 255
    return mag.astype(np.uint8)

def canny_auto(ch, mascara=None, sigma=0.33):
    """Canny con umbrales derivados de la mediana de la región de interés."""
    vals = ch[mascara > 0] if mascara is not None and (mascara > 0).any() else ch.ravel()
    v = np.median(vals)
    lo = int(max(0, (1.0 - sigma) * v))
    hi = int(min(255, (1.0 + sigma) * v))
    return cv2.Canny(ch, lo, hi), (lo, hi)

def bordes_de(bgr, mascara, metodo='canny', canal=None):
    """Calcula bordes sobre la imagen COMPLETA y recorta con la máscara.

    Se enmascara al final para conservar el contorno real de la hoja
    y eliminar solo el fondo lejano (ver sección 2.1).
    """
    ch = canal_analisis(bgr, canal)
    if metodo == 'canny':
        e, th = canny_auto(ch, mascara)
    elif metodo == 'sobel':
        e, th = sobel_magnitud(ch), None
    else:
        raise ValueError("método debe ser 'canny' o 'sobel'")
    return cv2.bitwise_and(e, e, mask=mascara), th

# Prueba
ruta, clase, bgr, mascara = muestra_procesada[1]
_, th = bordes_de(bgr, mascara, 'canny')
print("Umbrales de Canny para", etiqueta_corta(clase), "->", th)

In [ ]:
# --- Figura 3.4: Sobel vs Canny por clase ---
fig, axes = plt.subplots(len(muestra_procesada), 4,
                         figsize=(15.5, 3.9 * len(muestra_procesada)))
if len(muestra_procesada) == 1:
    axes = axes[np.newaxis, :]

for i, (ruta, clase, bgr, mascara) in enumerate(muestra_procesada):
    sob, _ = bordes_de(bgr, mascara, 'sobel')
    can, _ = bordes_de(bgr, mascara, 'canny')
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    overlay = rgb.copy()
    overlay[can > 0] = [255, 0, 0]

    panels = [(rgb, "Original", None), (sob, "Sobel (magnitud)", 'gray'),
              (can, "Canny", 'gray'), (overlay, "Canny sobre original", None)]
    for j, (img, t, cm) in enumerate(panels):
        axes[i, j].imshow(img, cmap=cm)
        axes[i, j].set_title(t if i == 0 else "", fontsize=11)
        axes[i, j].axis('off')
    axes[i, 0].text(-0.07, 0.5, etiqueta_corta(clase), transform=axes[i, 0].transAxes,
                    rotation=90, va='center', ha='center', fontsize=9)

plt.suptitle("3.4 - Detección de bordes: Sobel vs. Canny (canal " + CANAL + ")",
             fontsize=15, y=1.002)
plt.tight_layout()
guardar("02_bordes_sobel_canny.png")
plt.show()

In [ ]:
# --- Densidad de borde interior: ¿las hojas enfermas generan más bordes internos? ---
# Se excluye una franja junto al contorno para no contar el borde de la hoja.

def interior_hoja(mascara_aj, margen=12):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (margen * 2 + 1,) * 2)
    return cv2.erode(mascara_aj, kernel, iterations=1)

print(f"{'Clase':<26}{'Sana':<7}{'Densidad borde interior':>26}")
print("-" * 60)
dens_filas = []
for ruta, clase, bgr, mascara in muestra_procesada:
    can, _ = bordes_de(bgr, mascara, 'canny')
    inner = interior_hoja(ajustar_mascara(mascara))
    n = (inner > 0).sum()
    d = (can[inner > 0] > 0).mean() * 100 if n > 0 else 0.0
    dens_filas.append((etiqueta_corta(clase), es_sana(clase), d))
    print(f"{etiqueta_corta(clase):<26}{'si' if es_sana(clase) else 'no':<7}{d:>24.2f} %")

## 4. Sección 3.5 — Detección de esquinas

**Harris.** Construye la matriz de autocorrelación M a partir de los gradientes en una
ventana local y evalúa `R = det(M) - k*traza(M)^2`. Los autovalores de M describen la
variación de intensidad: ambos grandes indica esquina, uno solo grande indica borde,
ambos pequeños indica región plana.

**Shi-Tomasi.** Misma matriz M, pero la respuesta es directamente `min(lambda1, lambda2)`.
Al no depender del parámetro empírico `k` resulta más estable, y `goodFeaturesToTrack`
devuelve las esquinas ya ordenadas por calidad y separadas por una distancia mínima.

La máscara ceñida se pasa como argumento para buscar puntos solo dentro de la hoja.

In [ ]:
def harris_puntos(ch, mascara=None, k=0.04, umbral_rel=0.01):
    """Devuelve coordenadas Nx2 [x, y] de la respuesta de Harris."""
    dst = cv2.cornerHarris(np.float32(ch), blockSize=2, ksize=3, k=k)
    dst = cv2.dilate(dst, None)              # realza máximos locales
    picos = dst > umbral_rel * dst.max()
    if mascara is not None:
        picos &= (mascara > 0)
    ys, xs = np.where(picos)
    return np.column_stack([xs, ys])

def shi_tomasi_puntos(ch, mascara=None, max_esquinas=250, calidad=0.01, dist_min=8):
    pts = cv2.goodFeaturesToTrack(ch, maxCorners=max_esquinas, qualityLevel=calidad,
                                  minDistance=dist_min, mask=mascara)
    if pts is None:
        return np.empty((0, 2), dtype=int)
    return pts.reshape(-1, 2).astype(int)

ruta, clase, bgr, mascara = muestra_procesada[1]
ch = canal_analisis(bgr)
maj = ajustar_mascara(mascara)
print("Harris:", len(harris_puntos(ch, maj)), "px de respuesta | Shi-Tomasi:",
      len(shi_tomasi_puntos(ch, maj)), "esquinas")

In [ ]:
# --- Figura 3.5: esquinas superpuestas sobre la imagen original ---
fig, axes = plt.subplots(len(muestra_procesada), 3,
                         figsize=(12.5, 4.1 * len(muestra_procesada)))
if len(muestra_procesada) == 1:
    axes = axes[np.newaxis, :]

for i, (ruta, clase, bgr, mascara) in enumerate(muestra_procesada):
    ch = canal_analisis(bgr)
    maj = ajustar_mascara(mascara)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    pts_h = harris_puntos(ch, maj)
    pts_s = shi_tomasi_puntos(ch, maj)

    for j, t in enumerate(["Original", "Harris", "Shi-Tomasi"]):
        axes[i, j].imshow(rgb)
        axes[i, j].set_title(t if i == 0 else "", fontsize=11)
        axes[i, j].axis('off')
    if len(pts_h):
        axes[i, 1].scatter(pts_h[:, 0], pts_h[:, 1], s=1.6, c='red', alpha=0.55)
    if len(pts_s):
        axes[i, 2].scatter(pts_s[:, 0], pts_s[:, 1], s=26, facecolors='none',
                           edgecolors='cyan', linewidths=1.3)
    axes[i, 0].text(-0.07, 0.5, etiqueta_corta(clase), transform=axes[i, 0].transAxes,
                    rotation=90, va='center', ha='center', fontsize=9)

plt.suptitle("3.5 - Detección de esquinas: Harris vs. Shi-Tomasi", fontsize=15, y=1.002)
plt.tight_layout()
guardar("03_esquinas_harris_shitomasi.png")
plt.show()

## 5. Análisis cuantitativo: ¿contorno o lesión?

El enunciado pide discutir si los puntos característicos se asocian **al contorno de la
hoja** o **a regiones afectadas por enfermedades**. En lugar de responder solo por
inspección visual, se clasifica cada esquina según su distancia al borde.

Se aplica la transformada de distancia sobre la máscara: para cada píxel devuelve la
distancia euclidiana al fondo más cercano. Una esquina a menos de `UMBRAL_BORDE`
píxeles del fondo se considera **de contorno**; el resto son **interiores**, candidatas
a corresponder a lesiones, ya que el interior de una hoja sana es relativamente
uniforme salvo por las nervaduras.

In [ ]:
UMBRAL_BORDE = 15   # px

def clasificar_esquinas(pts, mascara_aj, umbral=UMBRAL_BORDE):
    if len(pts) == 0:
        return 0, 0
    dist = cv2.distanceTransform((mascara_aj > 0).astype(np.uint8), cv2.DIST_L2, 5)
    h, w = dist.shape
    d = dist[np.clip(pts[:, 1], 0, h - 1), np.clip(pts[:, 0], 0, w - 1)]
    n_c = int((d < umbral).sum())
    return n_c, len(pts) - n_c

filas = []
for ruta, clase, bgr, mascara in muestra_procesada:
    maj = ajustar_mascara(mascara)
    pts = shi_tomasi_puntos(canal_analisis(bgr), maj)
    n_c, n_i = clasificar_esquinas(pts, maj)
    filas.append({"clase": etiqueta_corta(clase), "sana": es_sana(clase),
                  "total": n_c + n_i, "contorno": n_c, "interior": n_i,
                  "pct_interior": 100 * n_i / max(n_c + n_i, 1)})

print(f"{'Clase':<26}{'Total':>7}{'Contorno':>10}{'Interior':>10}{'% interior':>12}")
print("-" * 65)
for f in filas:
    print(f"{f['clase']:<26}{f['total']:>7}{f['contorno']:>10}"
          f"{f['interior']:>10}{f['pct_interior']:>11.1f}%")

In [ ]:
# --- Gráfico de barras: composición de las esquinas por clase ---
etiquetas = [f["clase"] for f in filas]
contorno  = [f["contorno"] for f in filas]
interior  = [f["interior"] for f in filas]
x = np.arange(len(etiquetas))

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.bar(x, contorno, 0.6, label="Contorno (< " + str(UMBRAL_BORDE) + " px del borde)",
       color="#4C72B0")
ax.bar(x, interior, 0.6, bottom=contorno, label="Interior (posible lesión)",
       color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(etiquetas, rotation=18, ha='right')
ax.set_ylabel("Número de esquinas (Shi-Tomasi)")
ax.set_title("Distribución de puntos característicos por región de la hoja")
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
guardar("04_analisis_esquinas.png")
plt.show()

## 6. Conclusiones

> **Completar tras ejecutar el notebook.** Las observaciones de abajo son las
> esperables; hay que contrastarlas con los números reales y corregir lo que no cuadre.

### Decisiones de diseño

- **Enmascarado posterior.** Enmascarar la imagen antes de Canny produce un contorno
  doble, porque la máscara del pipeline está dilatada ~4 px respecto a la hoja. Calcular
  los bordes sobre la imagen completa y recortar el resultado elimina el artefacto.
- **Canal a\* en lugar de escala de grises.** El verde sano y el marrón de las lesiones
  tienen luminancia similar (Δ ≈ 14 niveles para *early blight*), por lo que en gris
  Canny no las detecta. En a\* el contraste sube a ~39. Esta es probablemente la
  decisión que más impacto tuvo en los resultados.

### Bordes (3.4)

- **Sobel vs. Canny.** Sobel entrega respuesta continua y bordes anchos; Canny, por la
  supresión de no-máximos, produce contornos de un píxel. Para delimitar lesiones Canny
  es preferible porque cierra regiones.
- **Densidad de borde interior.** _(revisar la tabla)_ Se espera que las clases con
  manchas focales (Septoria, Early blight) muestren mayor densidad interna que la hoja
  sana, cuya respuesta interior se limita a las nervaduras.

### Esquinas (3.5)

- **Harris vs. Shi-Tomasi.** Harris devuelve regiones densas de respuesta que requieren
  umbralización; Shi-Tomasi entrega un conjunto disperso y ordenado por calidad, más
  adecuado para el análisis cuantitativo.
- **Localización de los puntos.** _(revisar tabla y gráfico)_ Buena parte de las
  esquinas se concentra en el contorno dentado de la hoja, la zona de mayor variación.
  Las esquinas interiores tienden a coincidir con los límites de las manchas necróticas.
- **Implicación.** Si el porcentaje de esquinas interiores separa hojas sanas de
  enfermas, ese conteo es una señal discriminativa. Aunque los descriptores de la
  práctica son LBP, HOG y BoW, esto respalda que la textura local es informativa.

### Limitaciones

- Una imagen por clase basta para las figuras, no para conclusiones estadísticas.
  Subir `N_POR_CLASE` permitiría reportar promedios y desviaciones.
- La separación contorno/interior depende de `UMBRAL_BORDE`; conviene comprobar que las
  conclusiones se sostienen con otros valores.
- La máscara viene de Otsu sobre escala de grises y puede fallar cuando la luminancia de
  la hoja se aproxima a la del fondo.